# Act 4 — Change over time

> **Opening question:** *"The Ames dataset runs from 2006 to 2010 — right through the worst housing crash in a generation. Can we see the crash in the data? And which neighbourhoods held their value best when the market collapsed?"*

---
This act turns static numbers into a film — watching prices move year by year
and pinpointing exactly when the market shifted.

In [ ]:
import sys
sys.path.append('..')
from src.utils import *
import plotly.graph_objects as go
from plotly.subplots import make_subplots

act_header(
    act_num=4,
    title="Change over time",
    opening_question="Did the 2008 financial crisis leave a mark on Ames house prices — and when exactly did things turn?"
)

df = load_processed('cleaned.csv')

## 4.1 — Median sale price by year sold
First: did prices actually fall in Ames during the financial crisis?

In [ ]:
if 'YrSold' in df.columns:
    yearly = df.groupby('YrSold')['SalePrice'].agg(['median', 'mean', 'count']).reset_index()
    yearly.columns = ['Year', 'Median', 'Mean', 'Count']

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=('Median sale price by year', 'Number of sales by year'),
        vertical_spacing=0.12
    )

    fig.add_trace(go.Scatter(
        x=yearly['Year'], y=yearly['Median'],
        mode='lines+markers+text',
        text=[f'${v/1000:.0f}k' for v in yearly['Median']],
        textposition='top center',
        line=dict(color='#534AB7', width=3),
        marker=dict(size=10, color='#534AB7'),
        name='Median price'
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=yearly['Year'], y=yearly['Count'],
        marker_color='#9FE1CB',
        name='Sales volume'
    ), row=2, col=1)

    # Annotate 2008 crisis
    fig.add_vline(x=2008, line_dash='dash', line_color='#D85A30', line_width=1.5)
    fig.add_annotation(
        x=2008, y=yearly['Median'].max() * 1.05,
        text='2008 financial crisis', font=dict(color='#D85A30', size=11),
        showarrow=False, row=1, col=1
    )

    fig.update_layout(
        title='Ames housing market 2006–2010: price and volume over time',
        title_font_size=14,
        template='plotly_white',
        showlegend=False,
        height=500,
        margin=dict(t=70, b=40)
    )
    fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=1, col=1)

    save_plotly(fig, 'act4_price_over_time.html')
    fig.show()

    insight_callout(
        "Ames was more resilient than the national market during 2008–2009.\n"
        "Sales volume dropped sharply — fewer people were buying — but median prices\n"
        "held relatively steady. This is the 'university town effect': Iowa State University\n"
        "provides a stable economic anchor that pure property markets don't have.",
        label="Why Ames survived the crash"
    )

## 4.2 — Which neighbourhoods held value best?
During a downturn, not all areas fall equally. Who were the safe havens?

In [ ]:
if 'Neighborhood' in df.columns and 'YrSold' in df.columns:
    nbr_year = df.groupby(['Neighborhood', 'YrSold'])['SalePrice'].median().reset_index()

    # Only show neighbourhoods with data in every year
    year_counts = nbr_year.groupby('Neighborhood')['YrSold'].count()
    full_nbrs = year_counts[year_counts == year_counts.max()].index.tolist()[:8]

    filtered = nbr_year[nbr_year['Neighborhood'].isin(full_nbrs)]

    fig = px.line(
        filtered,
        x='YrSold', y='SalePrice',
        color='Neighborhood',
        title='Median price by neighbourhood — who held value through the crisis?',
        labels={'YrSold': 'Year sold', 'SalePrice': 'Median sale price ($)'},
        template='plotly_white',
        color_discrete_sequence=CATEGORICAL,
        markers=True
    )
    fig.update_layout(
        title_font_size=13,
        yaxis_tickprefix='$',
        yaxis_tickformat=',.0f',
        margin=dict(t=60, b=40),
        hovermode='x unified'
    )
    save_plotly(fig, 'act4_neighborhood_trends.html')
    fig.show()

## 4.3 — Building boom eras: when were houses built vs when they sold?

In [ ]:
if 'YearBuilt' in df.columns:
    df['Era'] = pd.cut(
        df['YearBuilt'],
        bins=[0, 1939, 1959, 1979, 1999, 2010],
        labels=['Pre-1940', '1940–1959', '1960–1979', '1980–1999', '2000+']
    )

    era_price = df.groupby('Era', observed=True)['SalePrice'].median()

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(era_price.index, era_price.values / 1000,
                  color=[PALETTE['gray'], PALETTE['blue'], PALETTE['teal'],
                         PALETTE['purple'], PALETTE['coral']],
                  alpha=0.88, edgecolor='white', linewidth=0.6)
    ax.set_title('Do newer houses sell for more? Median price by building era', fontsize=13)
    ax.set_ylabel('Median sale price ($k)')
    ax.set_xlabel('Era built')
    for bar, val in zip(bars, era_price.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'${val/1000:.0f}k', ha='center', fontsize=10)
    plt.tight_layout()
    save_figure(fig, 'act4_price_by_era.png')
    plt.show()

    insight_callout(
        "Newer construction commands a clear premium — homes built after 2000 sell for\n"
        f"roughly ${era_price.get('2000+', 0)/1000:.0f}k at median vs ${era_price.get('Pre-1940', 0)/1000:.0f}k for pre-1940 homes.\n"
        "But in Act 5 we'll see that a well-remodelled older home can beat a new one."
    )

---
## Act 4 — Closing punchline

In [ ]:
punchline(
    "The 2008 crisis hit sales volume hard but Ames prices barely flinched — "
    "a university town with a stable economy is a natural buffer. "
    "Newer homes earn a premium, and some neighbourhoods are simply more crash-proof than others. "
    "In Act 5, we meet the outliers — the houses that broke every pattern."
)

**Next → [Act 5 — The Outlier Plot](05_act5_outliers.ipynb)**